# AI Lab - Practical Question No. 18
## BFS vs DFS Performance Analyzer

- **Syllabus Area**: Uninformed Search Techniques
- **Difficulty**: 🟠 Advanced
- **Practical Type**: Simulation / Comparison
- **Core AI Concept**: Algorithm Comparison, Time/Space efficiency, Optimality vs Arbitrary Pathfinding
- **Python Concepts Tested**: Algorithm encapsulation, metric tracking, empirical benchmarking
- **Expected Algorithm / Technique**: Breadth-First Search (BFS) and Depth-First Search (DFS)
- **Recommended Python Libraries**: `time`, `matplotlib` (Optional Challenge)
- **Complexity Expectation**: Empirical timing and counting
- **Constraints / Conditions**: Use the exact same neighbor ordering (e.g., Up, Down, Left, Right)
- **Input Requirements**: Hardcoded 10x10 maze matrix
- **Sample Input**: Execute comparison
- **Sample Output**: `BFS Nodes Expanded: 45, Path Length: 12 | DFS Nodes Expanded: 14, Path Length: 22`
- **University Practical Importance**: High
- **Optional Challenge**: Plot the expansion process dynamically using matplotlib

---

### Problem Statement
Write a Python program to solve a predefined maze using both BFS and DFS. The program must compare the two algorithms by counting the exact number of nodes expanded by each before reaching the goal, highlighting BFS's optimality vs DFS's arbitrary pathfinding.

### 1. Theoretical Foundations & Algorithmic Comparison

| Feature / Metric | Breadth-First Search (BFS) | Depth-First Search (DFS) |
| :--- | :--- | :--- |
| **Frontier Structure** | FIFO Queue (`collections.deque`) | LIFO Stack (`list.pop()` or recursion) |
| **Exploration Bias** | Concentric spherical waves (shallowest first) | Deep branch plunge until dead-end or cycle |
| **Time Complexity** | $\mathcal{O}(b^d)$ | $\mathcal{O}(b^m)$ |
| **Space Complexity** | $\mathcal{O}(b^d)$ *(Exponential RAM bottleneck)* | $\mathcal{O}(b \cdot m)$ *(Linear active branch)* |
| **Optimality Guarantee** | **Guaranteed Optimal** (Shortest path for unit cost) | **Non-Optimal** (Returns first arbitrary path found) |
| **Completeness** | **Yes** (if branching factor $b$ is finite) | **Yes** on finite state spaces with visited tracking |

#### The Neighbor Ordering Constraint
Both algorithms evaluate candidate moves in the exact same priority order:
1. **Up** $(-1, 0)$
2. **Down** $(+1, 0)$
3. **Left** $(0, -1)$
4. **Right** $(0, +1)$

To ensure that the iterative DFS LIFO stack pops neighbors in the strict forward order matching recursive DFS (`Up -> Down -> Left -> Right`), candidate successors are pushed onto the stack in **reversed order** (`reversed(successors)`).

In [ ]:
import sys
from pathlib import Path

# Ensure project root is in sys.path
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.bfs_dfs_performance_analyzer import (
    DEFAULT_10X10_MAZE,
    DEFAULT_START_COORD,
    DEFAULT_GOAL_COORD,
    DFS_FAVORED_10X10_MAZE,
    DFS_FAVORED_GOAL_COORD,
    CORNER_10X10_MAZE,
    CORNER_GOAL_COORD,
    OPEN_10X10_MAZE,
    UNSOLVABLE_10X10_MAZE,
    SAMPLE_OUTPUT_TARGET_STRING,
    solve_maze_bfs,
    solve_maze_dfs,
    compare_maze_solvers,
    plot_bfs_vs_dfs_expansion,
    get_viva_reference,
)

print("BFS vs DFS Performance Analyzer module imported successfully!")

### 2. Execution on the Default 10x10 Benchmark Maze
We run both BFS and DFS on the hardcoded 10x10 maze from Start `(0, 0)` to Goal `(5, 7)`.
- **BFS**: Explores 45 nodes and finds the strictly shortest, optimal path (12 steps).
- **DFS**: Wanders down a deep branch, exploring 40 nodes and returning an arbitrary, circuitous path (22 steps).

In [ ]:
comp_default = compare_maze_solvers(
    DEFAULT_10X10_MAZE,
    start=DEFAULT_START_COORD,
    goal=DEFAULT_GOAL_COORD,
)

print("=" * 78)
print("  SAMPLE OUTPUT MATCH:")
print(f"  Target  : {SAMPLE_OUTPUT_TARGET_STRING}")
print(f"  Live Run: {comp_default.sample_output_formatted}")
print("=" * 78)
print("\nCOMPARATIVE METRICS TABLE:")
print(comp_default.format_summary_table())

### 3. ASCII Maze Path Visualization
Notice how BFS finds a clean, direct L-shaped path, whereas DFS takes an arbitrary detour all the way down to row 9 before ascending back to Goal `(5, 7)`.

In [ ]:
print("--- BFS Path (Optimal Shortest Steps: 12): ---")
print(comp_default.bfs.render_ascii_path())
print(f"BFS Action Moves: {comp_default.bfs.actions_str}")

print("\n--- DFS Path (Arbitrary Suboptimal Steps: 22): ---")
print(comp_default.dfs.render_ascii_path())
print(f"DFS Action Moves: {comp_default.dfs.actions_str}")

### 4. Optional Challenge: Dynamic Expansion Process Heatmaps (Matplotlib)
We plot the expansion process showing:
1. **BFS Heatmap**: Concentric waves expanding outward from Start `(0, 0)` with the shortest path overlayed.
2. **DFS Heatmap**: Deep single-branch penetration following neighbor priorities with the arbitrary path overlayed.
3. **Metrics Bar Chart**: Side-by-side comparison of Nodes Expanded, Path Length, and Memory (Peak Frontier Size).

In [ ]:
%matplotlib inline
fig = plot_bfs_vs_dfs_expansion(
    comp_default,
    save_path="../reports/bfs_vs_dfs_performance.png",
    show=True,
    title_suffix="Default 10x10 Benchmark Maze",
)

### 5. Viva Voce Scenario: Where DFS Outperforms BFS in Expansions
When the goal happens to reside deep along the very branch explored first by the neighbor ordering, DFS dives straight to the goal in only **14 node expansions**, whereas BFS must expand **45 nodes** across all branches to reach depth 12.

In [ ]:
comp_favored = compare_maze_solvers(
    DFS_FAVORED_10X10_MAZE,
    start=(0, 0),
    goal=DFS_FAVORED_GOAL_COORD,
)

print(f"Live Output: {comp_favored.sample_output_formatted}")
print(comp_favored.format_summary_table())

### 6. Viva Voce Questions & Authoritative Answers

#### Q1: Why is BFS guaranteed to find the optimal path in grid mazes, while DFS is not?
- **Answer**: In a grid maze where all orthogonal transitions have uniform unit cost ($c = 1$), path cost $g(n)$ equals node depth $d$. BFS uses a FIFO queue, guaranteeing all nodes at depth $d$ are expanded before any node at depth $d + 1$. Thus, the first goal reached is guaranteed to have minimal steps. DFS uses a LIFO stack, plunging down one path without depth synchronization, returning whatever arbitrary path hits the goal first.

#### Q2: In what scenarios does DFS outperform BFS?
- **Answer**:
  1. **Goal in early deep branch**: When the goal lies along the first subtree searched, DFS reaches it in $\mathcal{O}(d)$ expansions without exploring sibling branches, whereas BFS must expand $\mathcal{O}(b^d)$ nodes.
  2. **Memory limits**: BFS requires $\mathcal{O}(b^d)$ exponential memory; DFS requires only $\mathcal{O}(b \cdot m)$ linear memory.
  3. **Dense solutions**: When many solutions exist and any valid path is acceptable.

#### Q3: What is the impact of the branching factor $b$?
- **Answer**: Branching factor $b$ is the average number of valid moves per state ($1 \le b \le 4$ on grid mazes). Increasing $b$ causes BFS memory $\mathcal{O}(b^d)$ to explode exponentially, while DFS memory $\mathcal{O}(b \cdot m)$ grows only linearly.